# KCV-TTS Finetuning di Kaggle (2x T4, DDP, W&B)
Notebook ini menyiapkan pipeline penuh: venv, install dependency (termasuk `causal-conv1d` dan `mamba-ssm`), clone repo, prepare dataset CSV -> Arrow, lalu finetuning dengan 2 GPU T4 menggunakan `accelerate` + W&B.

## 1) Isi Konfigurasi
Ubah nilai di cell berikut sesuai repo dan path dataset kamu di Kaggle.

In [1]:
import os
from pathlib import Path

# ===== WAJIB DIISI =====
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"
WANDB_API_KEY = "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"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kcvanguard"

# CSV metadata sumber di Kaggle (header: audio_file|text, path audio absolut)
# Contoh: /kaggle/input/your-dataset/metadata.csv
SRC_METADATA_CSV = "/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv"

# Nama dataset hasil prep (akan jadi data/{DATASET_NAME}_pinyin)
DATASET_NAME = "datasetku"

# Output training
MODEL_NAME = "F5TTS_MAMBA_kaggle_full"
TARGET_SAMPLE_RATE = 24000

# Training optimization untuk 2x T4
BATCH_SIZE_PER_GPU = 512
MAX_SAMPLES = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1.0e-5
NUM_WARMUP_UPDATES = 200
EPOCHS = 50
NUM_WORKERS = 4

# Path kerja
WORKDIR = Path('/kaggle/temp')
VENV_DIR = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'

assert WANDB_API_KEY != "PASTE_WANDB_API_KEY_DI_SINI", "Isi WANDB_API_KEY dulu"
assert REPO_URL != "https://github.com/<username>/<repo>.git", "Isi REPO_URL dulu"
assert Path(SRC_METADATA_CSV).exists(), f"metadata tidak ditemukan: {SRC_METADATA_CSV}"

# Export ke environment agar tersedia di semua cell %%bash
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online'
os.environ['SRC_METADATA_CSV'] = SRC_METADATA_CSV
os.environ['DATASET_NAME'] = DATASET_NAME
os.environ['MODEL_NAME'] = MODEL_NAME
os.environ['TARGET_SAMPLE_RATE'] = str(TARGET_SAMPLE_RATE)
os.environ['BATCH_SIZE_PER_GPU'] = str(BATCH_SIZE_PER_GPU)
os.environ['MAX_SAMPLES'] = str(MAX_SAMPLES)
os.environ['GRAD_ACCUM'] = str(GRAD_ACCUM)
os.environ['LEARNING_RATE'] = str(LEARNING_RATE)
os.environ['NUM_WARMUP_UPDATES'] = str(NUM_WARMUP_UPDATES)
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['NUM_WORKERS'] = str(NUM_WORKERS)

print('Config OK')
print('SRC_METADATA_CSV =', SRC_METADATA_CSV)

Config OK
SRC_METADATA_CSV = /kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv


In [2]:
!nvidia-smi

Fri Mar 27 03:04:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Build Venv + Clone Repo + Install Dependencies

In [3]:
%%bash
apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils > /dev/null
python3.11 --version

Python 3.11.15


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%bash
set -euo pipefail
mkdir -p /kaggle/temp
cd /kaggle/temp

# Isolate from host/site customizations that can break venv bootstrap.
export PYTHONNOUSERSITE=1
unset PYTHONPATH || true
unset PYTHONHOME || true

# Require Python 3.11 to match local ABI/wheels exactly.
if command -v python3.11 >/dev/null 2>&1; then
  PY311=python3.11
else
  echo "python3.11 tidak tersedia di runtime ini. Ganti Kaggle image/runtime yang punya Python 3.11."
  exit 1
fi

# Build venv (fallback to virtualenv if stdlib venv fails).
if ! "$PY311" -m venv .venv; then
  "$PY311" -m pip install --upgrade pip
  "$PY311" -m pip install virtualenv
  "$PY311" -m virtualenv .venv
fi

source .venv/bin/activate
python - <<'PY'
import sys
assert sys.version_info[:2] == (3, 11), f"Butuh Python 3.11, dapat: {sys.version}"
print('python =', sys.version.split()[0])
PY

python -m pip install --upgrade pip setuptools wheel wrapt

# Clone repo
: "${REPO_BRANCH:=main}"
: "${REPO_URL:?REPO_URL is not set. Run Cell 3 (config) first.}"
if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" kcv-tts

cd kcv-tts

# Match stack exactly: torch 2.6/cu124 + project deps
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps

python = 3.11.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 79.0.1
    Uninstalling setuptools-79.0.1:
      Successfully uninstalled setuptools-79.0.1
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing met

Cloning into 'kcv-tts'...


In [5]:
# Requested order: causal first, then mamba
# Requested order: causal first, then mamba
!/kaggle/temp/.venv/bin/pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.3/287.3 MB 27.4 MB/s  0:00:10


In [6]:
!/kaggle/temp/.venv/bin/pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.3/533.3 MB 28.4 MB/s  0:00:15


In [7]:
%%bash
set -e

VENV=/kaggle/temp/.venv

"$VENV/bin/python" -V
"$VENV/bin/python" -c "import sys; print(sys.executable)"
"$VENV/bin/pip" -V

"$VENV/bin/pip" install --force-reinstall numpy==1.26.4 scipy==1.13.1

"$VENV/bin/python" - <<'PY'
from importlib import metadata
import sys

print("PYTHON:", sys.executable)
print("VERSION:", sys.version)

pkgs = [
    'torch','torchaudio','torchvision',
    'numpy','scipy','accelerate','hydra-core',
    'causal-conv1d','mamba-ssm','bitsandbytes'
]
for p in pkgs:
    try:
        print(p, metadata.version(p))
    except metadata.PackageNotFoundError:
        print(p, 'MISSING')
PY

Python 3.11.15
/kaggle/temp/.venv/bin/python
pip 26.0.1 from /kaggle/temp/.venv/lib/python3.11/site-packages/pip (python 3.11)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 39.8 MB/s  0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.17.1
    Uninstalling scipy-1.17.1:
      Successfully uninstalled scipy-1.17.1

PYTHON: /kaggle/temp/.venv/bin/python
VERSION: 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]
torch 2.6.0+cu124
torchaudio 2.6.0+cu124
torchvision 0.21.0+cu124
numpy 1.26.4
scipy 1.13.1
accelerate 1.13.0
hydra-core 1.3.2
causal-conv1d 1.6.1
mamba-ssm 2.3.1
bitsandbytes 0.49.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
f5-tts 1.1.18 requires gradio>=6.0.0, which is not installed.


In [8]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate
python - <<'PY'
import os
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('W&B login OK')
PY

W&B login OK


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3) Prepare Dataset (CSV -> Arrow)

In [9]:
import csv
from pathlib import Path

src = Path(SRC_METADATA_CSV)
dst = Path('/kaggle/temp/metadata_abs.csv')

with src.open('r', encoding='utf-8-sig', newline='') as fin, dst.open('w', encoding='utf-8', newline='') as fout:
    r = csv.reader(fin, delimiter='|')
    w = csv.writer(fout, delimiter='|', lineterminator='\n')
    header = next(r, None)
    if not header or header[0].strip() != 'audio_file' or header[1].strip() != 'text':
        raise ValueError('Header CSV harus: audio_file|text')
    w.writerow(['audio_file', 'text'])
    n = 0
    for row in r:
        if len(row) < 2:
            continue
        audio = row[0].strip()
        text = row[1].strip()
        if not audio or not text:
            continue
        p = Path(audio).expanduser()
        if not p.is_absolute():
            # Jika relatif, jadikan relatif terhadap folder CSV sumber
            p = (src.parent / p).resolve()
        if p.exists():
            w.writerow([p.as_posix(), text])
            n += 1

print('metadata_abs.csv rows =', n)
print('saved at', dst)

metadata_abs.csv rows = 4972
saved at /kaggle/temp/metadata_abs.csv


In [10]:
%%bash
set -e

TARGET_DIR="/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin"
TMP_DIR="/kaggle/temp/hf_tmp_emilia"

pip install -q huggingface_hub

python - <<'PY'
from huggingface_hub import snapshot_download
import os, shutil

target = "/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin"
tmpdir = "/kaggle/temp/hf_tmp_emilia"

snapshot_download(
    repo_id="mrfakename/E2-F5-TTS",
    repo_type="space",
    revision="27cee60c0890d22dab124730a73d5453fc8359a5",
    allow_patterns=["data/Emilia_ZH_EN_pinyin/*"],
    local_dir=tmpdir,
    local_dir_use_symlinks=False,
)

src = os.path.join(tmpdir, "data", "Emilia_ZH_EN_pinyin")
os.makedirs(target, exist_ok=True)

for name in os.listdir(src):
    s = os.path.join(src, name)
    d = os.path.join(target, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("done:", target)
PY

ls -lah "$TARGET_DIR"
test -f "$TARGET_DIR/vocab.txt"

done: /kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin
total 24K
drwxr-xr-x 2 root root 4.0K Mar 27 03:11 .
drwxr-xr-x 3 root root 4.0K Mar 27 03:11 ..
-rw-r--r-- 1 root root  14K Mar 27 03:11 vocab.txt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]


In [11]:
%%bash
set -euxo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

VENV_PY="/kaggle/temp/.venv/bin/python"
DATASET_NAME="${DATASET_NAME:-datasetku}"
META_CSV="/kaggle/temp/metadata_abs.csv"
OUT_DIR="/kaggle/temp/kcv-tts/data/${DATASET_NAME}_pinyin"

test -x "$VENV_PY"
test -f "$META_CSV"

echo "DATASET_NAME=$DATASET_NAME"
echo "META_CSV=$META_CSV"
echo "OUT_DIR=$OUT_DIR"
head -n 3 "$META_CSV"

"$VENV_PY" src/f5_tts/train/datasets/prepare_csv_wavs.py "$META_CSV" "$OUT_DIR" --workers 8
ls -lah "$OUT_DIR"

DATASET_NAME=datasetku
META_CSV=/kaggle/temp/metadata_abs.csv
OUT_DIR=/kaggle/temp/kcv-tts/data/datasetku_pinyin
audio_file|text
/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/wavs/KCVTTS-0000.wav|Reaksi berbeda disampaikan penasihat hukum pengungsi TimTim pro-integrasi Suhardi Somomoeljono.
/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/wavs/KCVTTS-0001.wav|Konflik eksekutif-legislatif sebenarnya adalah hal yang lumrah dalam kehidupan berdemokrasi.

Processing 4972 audio files using 8 workers...

Saving to /kaggle/temp/kcv-tts/data/datasetku_pinyin ...

For datasetku_pinyin, sample count: 4972
For datasetku_pinyin, vocab size is: 68
For datasetku_pinyin, total 6.44 hours
total 2.0M
drwxr-xr-x 2 root root 4.0K Mar 27 03:11 .
drwxr-xr-x 4 root root 4.0K Mar 27 03:11 ..
-rw-r--r-- 1 root root  38K Mar 27 03:11 duration.json
-rw-r--r-- 1 root root 1.9M Mar 27 03:11 raw.arrow
-rw-r--r-- 1 root root  14K Mar 27 03:11 vocab.txt


+ source /kaggle/temp/.venv/bin/activate
++ deactivate nondestructive
++ '[' -n '' ']'
++ '[' -n '' ']'
++ hash -r
++ '[' -n '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/kaggle/temp/.venv
++ export VIRTUAL_ENV
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/kaggle/temp/.venv/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' -n '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS1=
++ PS1='(.venv) '
++ export PS1
++ VIRTUAL_ENV_PROMPT='(.venv) '
++ export VIRTUAL_ENV_PROMPT
++ hash -r
+ cd /kaggle/temp/kcv-tts
+ VENV_PY=/kaggle/temp/.venv/bin/python
+ DATASET_NAME=datasetku
+ META_CSV=/kaggle/temp/metadata_abs.csv
+ OUT_DIR=/kaggle/temp/kcv-tts/data/datasetku_pinyin
+ test -x /kaggle/temp/.venv/bin/pyt

## 4) Download Pretrained Checkpoint ke Save Dir (untuk Finetune)

In [12]:
%%bash
set -euxo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

python - <<'PY'
from pathlib import Path
import os
import shutil

from huggingface_hub import hf_hub_download

# Force Indonesian warm-start assets from this repo
repo_id = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
ckpt_filename = "f5_tts_indo_v2.pt"
vocab_filename = "vocab.txt"

model_name = os.environ.get("MODEL_NAME", "F5TTS_MAMBA_kaggle_full")
dataset_name = os.environ.get("DATASET_NAME", "datasetku")
ckpt_dir = Path(f"/kaggle/working/ckpts/{model_name}_vocos_pinyin_{dataset_name}")
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Trainer auto-detects pretrained_* when no model_*.pt exists
dst_pt = ckpt_dir / "pretrained_indo_v2.pt"
src_pt = hf_hub_download(repo_id=repo_id, filename=ckpt_filename)
shutil.copy2(src_pt, dst_pt)

dst_vocab = ckpt_dir / "vocab.txt"
src_vocab = hf_hub_download(repo_id=repo_id, filename=vocab_filename)
shutil.copy2(src_vocab, dst_vocab)

print("repo:", repo_id)
print("checkpoint ready:", dst_pt)
print("vocab ready:", dst_vocab)
PY

ls -lah "$CKPT_DIR"

repo: Eempostor/F5-TTS-INDO-FINETUNE-V2
checkpoint ready: /kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/pretrained_indo_v2.pt
vocab ready: /kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/vocab.txt
total 1.3G
drwxr-xr-x 2 root root 4.0K Mar 27 03:11 .
drwxr-xr-x 3 root root 4.0K Mar 27 03:11 ..
-rw-r--r-- 1 root root 1.3G Mar 27 03:11 pretrained_indo_v2.pt
-rw-r--r-- 1 root root  14K Mar 27 03:11 vocab.txt


+ source /kaggle/temp/.venv/bin/activate
++ deactivate nondestructive
++ '[' -n '' ']'
++ '[' -n '' ']'
++ hash -r
++ '[' -n '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/kaggle/temp/.venv
++ export VIRTUAL_ENV
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/kaggle/temp/.venv/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' -n '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS1=
++ PS1='(.venv) '
++ export PS1
++ VIRTUAL_ENV_PROMPT='(.venv) '
++ export VIRTUAL_ENV_PROMPT
++ hash -r
+ cd /kaggle/temp/kcv-tts
+ DATASET_NAME=datasetku
+ MODEL_NAME=F5TTS_MAMBA_kaggle_full
+ CKPT_DIR=/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku
+ mkdir -p /kaggle/working/ckpts/F5TTS_MAMBA_kaggle_f

## 5) Finetune Full di 2x T4 (DDP) + W&B

In [13]:
# %%bash
# set -euo pipefail
# source /kaggle/temp/.venv/bin/activate

# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda)
# print("cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# pip uninstall -y causal-conv1d mamba-ssm || true
# pip cache purge || true

# # 1) Coba wheel yang match torch2.6 + cu12 + cp311 + cxx11abiTRUE
# pip install -U --no-deps \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
# || {
#   echo "[warn] direct wheel install gagal, fallback ke PyPI build"
#   pip install -U --no-build-isolation causal-conv1d==1.5.0.post8 mamba-ssm==2.3.1
# }

# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [14]:
# %%bash
# set -euo pipefail
# source /kaggle/temp/.venv/bin/activate

# echo "[A] cek torch ABI"
# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda, "cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# echo "[B] uninstall + bersihkan sisa file"
# pip uninstall -y causal-conv1d mamba-ssm || true
# python - <<'PY'
# import site, glob, os, shutil
# for d in site.getsitepackages():
#     for p in glob.glob(os.path.join(d, "causal_conv1d*")) + glob.glob(os.path.join(d, "mamba_ssm*")):
#         if os.path.isdir(p):
#             shutil.rmtree(p, ignore_errors=True)
#         else:
#             try: os.remove(p)
#             except: pass
#         print("removed:", p)
# PY
# pip cache purge || true

# echo "[C] toolchain"
# apt-get update -y
# apt-get install -y build-essential python3.11-dev

# echo "[D] build ulang paksa ABI FALSE"
# export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: causal-conv1d==1.5.0.post8
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: mamba-ssm==2.3.1

# echo "[E] verifikasi import"
# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__, "abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [15]:
!apt-get update -y
!apt-get install -y build-essential python3.11-dev

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



build-essential is already the newest ve

In [16]:
# %%bash
# set -Eeuo pipefail

# on_err() {
#   ec=$?
#   echo ""
#   echo "[ERROR] line=$1 exit=$ec"
#   if [ -f /kaggle/working/train_kaggle.log ]; then
#     echo "----- tail /kaggle/working/train_kaggle.log -----"
#     tail -n 200 /kaggle/working/train_kaggle.log || true
#   fi
#   exit "$ec"
# }
# trap 'on_err $LINENO' ERR

# source /kaggle/temp/.venv/bin/activate
# cd /kaggle/temp/kcv-tts

# export HYDRA_FULL_ERROR=1
# export TORCH_DISTRIBUTED_DEBUG=DETAIL
# export PYTHONUNBUFFERED=1

# DATASET_NAME="${DATASET_NAME:-datasetku}"
# MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
# BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-256}"
# MAX_SAMPLES="${MAX_SAMPLES:-1}"
# NUM_WORKERS="${NUM_WORKERS:-2}"
# EPOCHS="${EPOCHS:-1}"
# LEARNING_RATE="${LEARNING_RATE:-1e-5}"
# NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-20}"
# GRAD_ACCUM="${GRAD_ACCUM:-4}"
# TARGET_SAMPLE_RATE="${TARGET_SAMPLE_RATE:-24000}"
# WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

# LOG_FILE=/kaggle/working/train_kaggle.log
# : > "$LOG_FILE"

# echo "[1/6] runtime check"
# which python
# python -V
# which accelerate
# accelerate launch --help >/dev/null

# echo "[2/6] dependency check (core only)"
# python - <<'PY'
# import importlib, torch
# print("torch", torch.__version__, "cuda", torch.version.cuda, "cxx11abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# for m in ["accelerate", "hydra"]:
#     mod = importlib.import_module(m)
#     print("OK", m, getattr(mod, "__version__", "n/a"))
# print("deps_core_ok")
# PY

# echo "[3/6] mamba optional check"
# if python - <<'PY'
# import importlib, sys
# try:
#     importlib.import_module("causal_conv1d")
#     importlib.import_module("mamba_ssm")
#     print("mamba_stack_ok")
#     sys.exit(0)
# except Exception as e:
#     print("mamba_stack_off:", e)
#     sys.exit(1)
# PY
# then
#   USE_MAMBA=true
# else
#   USE_MAMBA=false
# fi
# echo "USE_MAMBA=$USE_MAMBA"

# echo "[4/6] dataset + pretrained"
# DATA_DIR="data/${DATASET_NAME}_pinyin"
# test -d "$DATA_DIR"

# CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
# PRETRAIN_FILE="${CKPT_DIR}/pretrained_model_1250000.safetensors"

# if [ ! -f "$PRETRAIN_FILE" ]; then
#   ALT_PRETRAIN="$(ls -1 ckpts/*_vocos_pinyin_${DATASET_NAME}/pretrained_*.safetensors 2>/dev/null | head -n 1 || true)"
#   if [ -n "$ALT_PRETRAIN" ]; then
#     mkdir -p "$CKPT_DIR"
#     cp -f "$ALT_PRETRAIN" "$PRETRAIN_FILE"
#     echo "fallback pretrained copied: $ALT_PRETRAIN -> $PRETRAIN_FILE"
#   fi
# fi
# test -f "$PRETRAIN_FILE"

# echo "[5/6] logger + gpu"
# if [ -n "${WANDB_API_KEY:-}" ]; then
#   export WANDB_API_KEY
#   export WANDB_ENTITY="${WANDB_ENTITY:-}"
#   export WANDB_PROJECT
#   export WANDB_MODE=online
#   LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}-kaggle")
# else
#   export WANDB_MODE=offline
#   LOGGER_ARGS=(++ckpts.logger=tensorboard)
# fi

# GPU_COUNT="$(python - <<'PY'
# import torch
# print(torch.cuda.device_count())
# PY
# )"
# echo "GPU_COUNT=$GPU_COUNT"

# if [ "$GPU_COUNT" -ge 2 ]; then
#   export CUDA_VISIBLE_DEVICES=0,1
#   DIST_ARGS=(--multi_gpu --num_processes 2)
# elif [ "$GPU_COUNT" -eq 1 ]; then
#   export CUDA_VISIBLE_DEVICES=0
#   DIST_ARGS=(--num_processes 1)
# else
#   echo "No GPU detected"
#   exit 1
# fi

# if [ "$USE_MAMBA" = true ]; then
#   MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
# else
#   MODEL_ARGS=(++model.arch.use_mamba=false)
# fi

# echo "[6/6] training"
# set +e
# accelerate launch "${DIST_ARGS[@]}" --mixed_precision=fp16 \
#   src/f5_tts/train/train.py \
#   --config-name F5TTS_SANITY_HYBRID_1K.yaml \
#   "++datasets.name=${DATASET_NAME}" \
#   "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
#   ++datasets.batch_size_type=frame \
#   "++datasets.max_samples=${MAX_SAMPLES}" \
#   "++datasets.num_workers=${NUM_WORKERS}" \
#   "++optim.epochs=${EPOCHS}" \
#   "++optim.learning_rate=${LEARNING_RATE}" \
#   "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
#   "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
#   ++optim.max_grad_norm=1.0 \
#   ++optim.bnb_optimizer=False \
#   ++ckpts.log_samples=False \
#   ++ckpts.save_per_updates=200 \
#   ++ckpts.last_per_updates=100 \
#   ++ckpts.keep_last_n_checkpoints=3 \
#   "++model.name=${MODEL_NAME}" \
#   ++model.tokenizer=pinyin \
#   "++model.mel_spec.target_sample_rate=${TARGET_SAMPLE_RATE}" \
#   ++model.arch.dim=512 \
#   ++model.arch.depth=10 \
#   ++model.arch.heads=8 \
#   ++model.arch.text_dim=512 \
#   ++model.arch.checkpoint_activations=True \
#   "${MODEL_ARGS[@]}" \
#   "${LOGGER_ARGS[@]}" \
#   2>&1 | tee "$LOG_FILE"
# RC=${PIPESTATUS[0]}
# set -e

# if [ "$RC" -ne 0 ]; then
#   echo "Training gagal RC=$RC"
#   tail -n 200 "$LOG_FILE" || true
#   exit "$RC"
# fi

# echo "Training selesai RC=0"
# find "$CKPT_DIR" -maxdepth 3 -type f -name "model*.pt" -print | sort || true

In [17]:
%%bash
set -Eeuo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

export HYDRA_FULL_ERROR=1
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export PYTHONUNBUFFERED=1
export CUDA_VISIBLE_DEVICES=0,1

# Training defaults (override from Cell 3 env if needed)
DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-512}"
MAX_SAMPLES="${MAX_SAMPLES:-2}"
NUM_WORKERS="${NUM_WORKERS:-4}"
EPOCHS="${EPOCHS:-10}"
LEARNING_RATE="${LEARNING_RATE:-1e-5}"
NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-200}"
GRAD_ACCUM="${GRAD_ACCUM:-4}"

# Save checkpoints to persistent storage
CKPT_ROOT="/kaggle/working/ckpts"
CKPT_DIR="${CKPT_ROOT}/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

# Enforce warm-start from HF Indonesian checkpoint
PRETRAIN_PT="${CKPT_DIR}/pretrained_indo_v2.pt"
if [ ! -f "$PRETRAIN_PT" ]; then
  echo "Missing pretrained checkpoint: $PRETRAIN_PT"
  echo "Run Cell 17 (download pretrained) first."
  exit 1
fi

# Remove old model checkpoints so trainer loads pretrained_indo_v2.pt instead of model_last.pt
FORCE_FRESH_FROM_PRETRAIN="${FORCE_FRESH_FROM_PRETRAIN:-1}"
if [ "$FORCE_FRESH_FROM_PRETRAIN" = "1" ]; then
  find "$CKPT_DIR" -maxdepth 1 -type f -name 'model_*.pt' -delete || true
  rm -f "$CKPT_DIR/model_last.pt" || true
  echo "Removed old model_*.pt/model_last.pt (fresh start from pretrained_indo_v2.pt)"
fi

# Keep tokenizer as pinyin for train.py dataset loader.
# train.py couples dataset path to tokenizer: data/{datasets.name}_{model.tokenizer}.
DATA_DIR="data/${DATASET_NAME}_pinyin"
DATA_RAW_ARROW="${DATA_DIR}/raw.arrow"
DATA_RAW_DIR="${DATA_DIR}/raw"
DATA_VOCAB="${DATA_DIR}/vocab.txt"
if [ ! -f "$DATA_RAW_ARROW" ] && [ ! -d "$DATA_RAW_DIR" ]; then
  echo "Dataset not found: need ${DATA_RAW_ARROW} or ${DATA_RAW_DIR}"
  exit 1
fi

# Sync Indo vocab from CKPT_DIR into dataset vocab path used by pinyin tokenizer.
if [ -f "${CKPT_DIR}/vocab.txt" ]; then
  cp -f "${CKPT_DIR}/vocab.txt" "$DATA_VOCAB"
  echo "Tokenizer vocab synced: ${CKPT_DIR}/vocab.txt -> ${DATA_VOCAB}"
fi
TOKENIZER_ARGS=(++model.tokenizer=pinyin)

echo "Disk status before train:"
df -h /kaggle/temp /kaggle/working || true

python - <<'PY'
import torch, accelerate, hydra
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu_count", torch.cuda.device_count())
print("accelerate", accelerate.__version__)
print("hydra", hydra.__version__)
PY

# Mamba optional fallback
if python - <<'PY'
import importlib, sys
try:
    importlib.import_module("causal_conv1d")
    importlib.import_module("mamba_ssm")
    sys.exit(0)
except Exception:
    sys.exit(1)
PY
then
  MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
  echo "USE_MAMBA=true"
else
  MODEL_ARGS=(++model.arch.use_mamba=false)
  echo "USE_MAMBA=false"
fi

if [ -n "${WANDB_API_KEY:-}" ]; then
  export WANDB_API_KEY
  export WANDB_ENTITY="${WANDB_ENTITY:-}"
  export WANDB_PROJECT
  export WANDB_MODE=online
  echo "Using logger: wandb"
  LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}")
else
  export WANDB_MODE=offline
  echo "Using logger: tensorboard"
  LOGGER_ARGS=(++ckpts.logger=tensorboard)
fi

# Disable periodic checkpointing; keep only final model_last.pt at end of training.
NO_PERIODIC_SAVE_UPDATES=999999999

accelerate launch --multi_gpu --num_processes 2 --mixed_precision=fp16 \
  src/f5_tts/train/train.py \
  --config-name F5TTS_SANITY_HYBRID_1K.yaml \
  "++datasets.name=${DATASET_NAME}" \
  "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
  ++datasets.batch_size_type=frame \
  "++datasets.max_samples=${MAX_SAMPLES}" \
  "++datasets.num_workers=${NUM_WORKERS}" \
  "++optim.epochs=${EPOCHS}" \
  "++optim.learning_rate=${LEARNING_RATE}" \
  "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
  "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
  ++optim.max_grad_norm=1.0 \
  ++optim.bnb_optimizer=False \
  ++ckpts.log_samples=False \
  "++ckpts.save_per_updates=${NO_PERIODIC_SAVE_UPDATES}" \
  "++ckpts.last_per_updates=${NO_PERIODIC_SAVE_UPDATES}" \
  ++ckpts.keep_last_n_checkpoints=0 \
  "++ckpts.save_dir=${CKPT_DIR}" \
  "++model.name=${MODEL_NAME}" \
  "${TOKENIZER_ARGS[@]}" \
  "${MODEL_ARGS[@]}" \
  "${LOGGER_ARGS[@]}"

echo "Done: ${CKPT_DIR}"
find "${CKPT_DIR}" -maxdepth 2 -type f -name 'model*.pt' -print | sort || true

Removed old model_*.pt/model_last.pt (fresh start from pretrained_indo_v2.pt)
Tokenizer vocab synced: /kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/vocab.txt -> data/datasetku_pinyin/vocab.txt
Disk status before train:
Filesystem      Size  Used Avail Use% Mounted on
overlay         7.9T  6.7T  1.2T  86% /
/dev/loop1       20G  1.3G   19G   7% /kaggle/working
torch 2.6.0+cu124 cuda 12.4 gpu_count 2
accelerate 1.13.0
hydra 1.3.2
USE_MAMBA=false
Using logger: wandb
Using logger: wandb
Gradient accumulation checkpointing with per_updates now, old logic per_steps used with before f992c4e
Loading dataset ...
Loading dataset ...
[Training Config Summary] epochs=50 lr=1e-05 warmup_updates=200 batch_size_per_gpu=512 batch_size_type=frame max_samples=2 grad_accumulation_steps=4 max_grad_norm=1.0
[CFM Experimental Summary] backbone=HybridDiT use_mamba=False mamba_layers=[10, 11] use_distill=False lambda_distill_out=0.000000 lambda_distill_hidden=0.000000 use_ctc=False lamb

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run kghwnuh5
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/temp/kcv-tts/wandb/run-20260327_031233-kghwnuh5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run F5TTS_MAMBA_kaggle_full
wandb: ⭐️ View project at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kcvanguard
wandb: 🚀 View run at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi

## 6) Lokasi Checkpoint dan Quick Inference (Opsional)

In [18]:
from pathlib import Path
import glob

candidates = [
    Path(f'/kaggle/working/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
    Path(f'/kaggle/temp/kcv-tts/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
    Path(f'/kaggle/temp/kcv-tts/kaggle/working/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
]

print('Candidate checkpoint dirs:')
for d in candidates:
    print('-', d, 'exists=', d.exists())
    pts = sorted(d.glob('model*.pt'))
    for p in pts[-5:]:
        print('  ->', p)

# Broad fallback scan
fallback_patterns = [
    '/kaggle/temp/kcv-tts/**/model_last.pt',
    '/kaggle/temp/kcv-tts/**/model_*.pt',
]

found = []
for pat in fallback_patterns:
    found.extend(glob.glob(pat, recursive=True))

found = sorted(set(found))
print('Fallback model checkpoints found:', len(found))
for p in found[-20:]:
    print('fallback ->', p)

Candidate checkpoint dirs:
- /kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku exists= True
- /kaggle/temp/kcv-tts/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku exists= True
- /kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku exists= True
  -> /kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/model_last.pt
Fallback model checkpoints found: 1
fallback -> /kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/model_last.pt


In [19]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate

mkdir -p /kaggle/working/ref_audio

python - <<'PY'
from huggingface_hub import hf_hub_download
import shutil

repo_id = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
files = [
    "ref_prabowo.mp3",
    "ref_reporter.mp3",
    "ref_windah.mp3",
]

for name in files:
    src = hf_hub_download(repo_id=repo_id, filename=name)
    dst = f"/kaggle/working/ref_audio/{name}"
    shutil.copy2(src, dst)
    print("downloaded:", dst)
PY

ls -lah /kaggle/working/ref_audio

downloaded: /kaggle/working/ref_audio/ref_prabowo.mp3
downloaded: /kaggle/working/ref_audio/ref_reporter.mp3
downloaded: /kaggle/working/ref_audio/ref_windah.mp3
total 244K
drwxr-xr-x 2 root root 4.0K Mar 27 09:08 .
drwxr-xr-x 4 root root 4.0K Mar 27 09:08 ..
-rw-r--r-- 1 root root  65K Mar 27 09:08 ref_prabowo.mp3
-rw-r--r-- 1 root root  57K Mar 27 09:08 ref_reporter.mp3
-rw-r--r-- 1 root root 105K Mar 27 09:08 ref_windah.mp3


In [20]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

# Avoid notebook inline backend leaking into CLI process.
unset MPLBACKEND || true
export MPLBACKEND=Agg

# Contoh quick inference setelah model_last.pt tersedia.
PRIMARY_CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_1="/kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_2="/kaggle/temp/kcv-tts/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"

# Fixed reference inputs requested
REF_AUDIO="/kaggle/working/ref_audio/ref_prabowo.mp3"
REF_TEXT="Selamat pagi, salam sejahtera bagi kita sekalian"

if [ ! -f "$REF_AUDIO" ]; then
  echo "Reference audio tidak ditemukan: $REF_AUDIO"
  echo "Jalankan dulu cell download reference audio."
  exit 1
fi

# Find model_last.pt from explicit and fallback locations.
MODEL_PT=""
for p in \
  "${PRIMARY_CKPT_DIR}/model_last.pt" \
  "${REL_CKPT_DIR_1}/model_last.pt" \
  "${REL_CKPT_DIR_2}/model_last.pt" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/model_last.pt 2>/dev/null | head -n 1 || true)" \
  "$(find /kaggle/temp/kcv-tts -type f -name model_last.pt 2>/dev/null | tail -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    MODEL_PT="$p"
    break
  fi
done

if [ -z "$MODEL_PT" ]; then
  echo "model_last.pt tidak ditemukan."
  echo "Cek cepat:"
  echo "  find /kaggle/temp/kcv-tts -type f -name 'model*.pt' | tail -n 50"
  exit 1
fi

# Derive checkpoint dir from model path to search nearby hydra config first.
MODEL_DIR="$(dirname "$MODEL_PT")"

# Find Hydra config from several possible training output locations.
MODEL_CFG=""
for p in \
  "$(ls -1t "${MODEL_DIR}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${PRIMARY_CKPT_DIR}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${REL_CKPT_DIR_1}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${REL_CKPT_DIR_2}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/*_${DATASET_NAME}/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    MODEL_CFG="$p"
    break
  fi
done

# Fallback: synthesize inference config from base training config when hydra output is missing.
if [ -z "$MODEL_CFG" ]; then
  echo "Hydra config tidak ditemukan, membuat fallback config dari base yaml..."
  MODEL_CFG="/kaggle/temp/infer_model_cfg.yaml"
  MODEL_PT="$MODEL_PT" MODEL_CFG="$MODEL_CFG" python - <<'PY'
import os
import torch
from omegaconf import OmegaConf

model_pt = os.environ["MODEL_PT"]
model_cfg_out = os.environ["MODEL_CFG"]
base_cfg = "src/f5_tts/configs/F5TTS_SANITY_HYBRID_1K.yaml"

cfg = OmegaConf.load(base_cfg)
ckpt = torch.load(model_pt, map_location="cpu", weights_only=True)
state = ckpt.get("ema_model_state_dict", ckpt.get("model_state_dict", {}))
keys = list(state.keys())
use_mamba = any("mamba" in k for k in keys)

cfg.model.arch.use_mamba = bool(use_mamba)
if not use_mamba:
    cfg.model.arch.mamba_layers = []

# Align identifiers used in this notebook run.
cfg.model.name = os.environ.get("MODEL_NAME", cfg.model.name)
cfg.datasets.name = os.environ.get("DATASET_NAME", cfg.datasets.name)
cfg.model.tokenizer = "pinyin"

OmegaConf.save(cfg, model_cfg_out)
print("fallback cfg saved:", model_cfg_out)
print("detected use_mamba:", use_mamba)
PY
fi

if [ ! -f "$MODEL_CFG" ]; then
  echo "Config yaml tetap tidak ditemukan/terbuat."
  exit 1
fi

echo "Using model: $MODEL_PT"
echo "Using config: $MODEL_CFG"
echo "Using ref audio: $REF_AUDIO"

python src/f5_tts/infer/infer_cli.py \
  -m F5TTS_Small \
  -mc "$MODEL_CFG" \
  -p "$MODEL_PT" \
  -v data/${DATASET_NAME}_pinyin/vocab.txt \
  -r "$REF_AUDIO" \
  -s "$REF_TEXT" \
  -t "Halo semua kenalin Aku Sedang Makan" \
  -o /kaggle/working \
  -w infer_kaggle.wav \
  --device cuda --nfe_step 24 --cfg_strength 2.0

Hydra config tidak ditemukan, membuat fallback config dari base yaml...
fallback cfg saved: /kaggle/temp/infer_model_cfg.yaml
detected use_mamba: False
Using model: /kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/model_last.pt
Using config: /kaggle/temp/infer_model_cfg.yaml
Using ref audio: /kaggle/working/ref_audio/ref_prabowo.mp3
Download Vocos from huggingface charactr/vocos-mel-24khz
Using F5TTS_Small...

vocab :  data/datasetku_pinyin/vocab.txt
token :  custom
model :  /kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/model_last.pt 

Voice: main
ref_audio  /kaggle/working/ref_audio/ref_prabowo.mp3
Converting audio...
Using custom reference text...

ref_text   Selamat pagi, salam sejahtera bagi kita sekalian. 
ref_audio_ /tmp/tmp6614kfso.wav 


No voice tag found, using main.
Voice: main
gen_text 0 Halo semua kenalin Aku Sedang Makan


Generating audio in 1 batches...
/kaggle/working/infer_kaggle.wav


100%|██████████| 1/1 [00:02<00:00,  2.49s/it]


In [21]:
%%bash
set -euo pipefail

MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
DATASET_NAME="${DATASET_NAME:-datasetku}"

PRIMARY_CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_1="/kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_2="/kaggle/temp/kcv-tts/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"

# Cari model_last.pt dari lokasi utama + fallback.
SRC_MODEL=""
for p in \
  "${PRIMARY_CKPT_DIR}/model_last.pt" \
  "${REL_CKPT_DIR_1}/model_last.pt" \
  "${REL_CKPT_DIR_2}/model_last.pt" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/model_last.pt 2>/dev/null | head -n 1 || true)" \
  "$(find /kaggle/temp/kcv-tts -type f -name model_last.pt 2>/dev/null | tail -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    SRC_MODEL="$p"
    break
  fi
done

if [ -z "$SRC_MODEL" ]; then
  echo "model_last.pt tidak ditemukan di semua lokasi kandidat."
  echo "Debug cepat:"
  echo "  find /kaggle/temp/kcv-tts -type f -name 'model*.pt' | tail -n 50"
  exit 1
fi

DST_DIR="/kaggle/working"
DST_MODEL="${DST_DIR}/model_last_finetuned.pt"
DST_VOCAB="${DST_DIR}/vocab_${DATASET_NAME}.txt"

cp -vf "$SRC_MODEL" "$DST_MODEL"
echo "Model copied to: $DST_MODEL"

SRC_DIR="$(dirname "$SRC_MODEL")"
if [ -f "${SRC_DIR}/vocab.txt" ]; then
  cp -vf "${SRC_DIR}/vocab.txt" "$DST_VOCAB"
  echo "Vocab copied to: $DST_VOCAB"
elif [ -f "${PRIMARY_CKPT_DIR}/vocab.txt" ]; then
  cp -vf "${PRIMARY_CKPT_DIR}/vocab.txt" "$DST_VOCAB"
  echo "Vocab copied from primary ckpt dir: $DST_VOCAB"
fi

echo ""
echo "Files ready in /kaggle/working:"
ls -lh "$DST_MODEL" || true
ls -lh "$DST_VOCAB" || true
ls -1 /kaggle/working/*.pt /kaggle/working/*.wav 2>/dev/null | head -n 50

'/kaggle/temp/kcv-tts/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/model_last.pt' -> '/kaggle/working/model_last_finetuned.pt'
Model copied to: /kaggle/working/model_last_finetuned.pt
'/kaggle/working/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/vocab.txt' -> '/kaggle/working/vocab_datasetku.txt'
Vocab copied from primary ckpt dir: /kaggle/working/vocab_datasetku.txt

Files ready in /kaggle/working:
-rw-r--r-- 1 root root 5.1G Mar 27 09:09 /kaggle/working/model_last_finetuned.pt
-rw-r--r-- 1 root root 14K Mar 27 09:09 /kaggle/working/vocab_datasetku.txt
/kaggle/working/infer_kaggle.wav
/kaggle/working/model_last_finetuned.pt
